# 1. MoSDeF Basics: Simple Systems

### What you will do here
- Build molecules in mBuild from SMILES strings and from files
- Pack them into a two phase system with a liquid liquid interface
- Hand the system to GMSO to create a `Topology`
- Load force fields, apply them, and inspect what changed in the GMSO `Topology`
- Write simulation inputs for LAMMPS, GROMACS, and HOOMD-Blue

This notebook is the whole pipeline end to end. Once you know the four steps
(build, convert, type, write) you can swap any piece out for another.

In [ ]:
import numpy as np

import mbuild as mb
import gmso
from gmso import ForceField
from gmso.parameterization import apply

# Quiet the library loggers so the workshop output stays readable
import logging
from mbuild import mBuildLogger
mBuildLogger().library_logger.setLevel(logging.ERROR)
gmso.gmso_logger.library_logger.setLevel(logging.ERROR)

## Loading a molecule from a SMILES string

A SMILES string is a short text code that describes the chemical structure of a molecule using standard alpha-numeric characters

`mb.load` with `smiles=True` creates an 
`mb.Compound` from a SMILES string with the correct 3D coordinates and a bond graph.

Below, you'll create an mBuild `Compound` of ethanol. The SMILES string for Ethanol is `CCO`

In [ ]:
ethanol = mb.load("CCO", smiles=True)
print(type(ethanol))
print(ethanol.n_particles, "particles,", ethanol.n_bonds, "bonds")
ethanol.visualize()

## Loading a molecule from a file
You can also create an mBuild compound from a file.

mBuild reads mol2, pdb, gro, xyz, and more. The result is the same kind of
object you just got from a SMILES string.

In [ ]:
benzene = mb.load("files/benzene.mol2")
print(type(benzene))
print(benzene.n_particles, "particles,", benzene.n_bonds, "bonds")
benzene.visualize()

## Building a multi-component system

mBuild wraps PACKMOL for putting molecules into a box. `mb.fill_box` fills a
box uniformly, `mb.solvate` puts a solute at the center and packs solvent
around it, and `mb.fill_region` gives each species its own sub-volume.

We want a water and hexane interface, so `fill_region` is the one we want. The
`bounds` argument takes `[min_x, min_y, min_z, max_x, max_y, max_z]` per
species, in nm. Water goes in the bottom of the box and hexane goes on top.

The `name` you give each Compound matters. GMSO carries it through as a
molecule label, and that is how you tell `apply` which force field goes with
which species.

In [ ]:
def n_from_density(rho, molar_mass, volume):
    """Molecules needed to fill volume (nm^3) at rho (g/cm^3)."""
    return int(rho / molar_mass * 6.022e23 * 1e-21 * volume)

water = mb.load("O", smiles=True)
water.name = "water"

hexane = mb.load("CCCCCC", smiles=True)
hexane.name = "hexane"

# 3.5 x 3.5 nm box, water from z=0 to 2.5, hexane from z=2.5 to 5.5
area, z_split, z_top = 3.5 * 3.5, 2.5, 5.5
n_water = n_from_density(0.997, 18.02, area * z_split)
n_hexane = n_from_density(0.655, 86.18, area * (z_top - z_split))
print(n_water, "water,", n_hexane, "hexane")

box = mb.Box([3.5, 3.5, z_top])
system = mb.fill_region(
    compound=[water, hexane],
    n_compounds=[n_water, n_hexane],
    region=box,
    bounds=[[0, 0, 0, 3.5, 3.5, z_split],
            [0, 0, z_split, 3.5, 3.5, z_top]],
    edge=0.15,
)
system.box = box
system.periodicity = (True, True, True)
print(system.n_particles, "particles")
system.visualize()

PACKMOL gives a deliberately sharp starting interface. Under MD it will relax
into a real interfacial profile, so do not read the packed structure as an
equilibrium one.

Note also that a periodic box with two stacked phases has two interfaces, one
in the middle and one across the z boundary. That is the standard way to set up
a slab geometry.

In [ ]:
for name in ("water", "hexane"):
    z = [float(np.mean(child.xyz[:, 2])) for child in system.children if child.name == name]
    print(f"{name:7s} n={len(z):5d}  z from {min(z):.2f} to {max(z):.2f} nm")

## Converting to a GMSO Topology

`Compound.to_gmso()` converts an mBuild `Compound` object into a GMSO `Topology` object.
At this point not force fields have been applied, or atom type names assigned. The topology knows
about sites, bonds, and molecule labels.

In [ ]:
topology = system.to_gmso()

print("sites     :", topology.n_sites)
print("bonds     :", topology.n_bonds)
print("angles    :", topology.n_angles)
print("dihedrals :", topology.n_dihedrals)
print("typed     :", topology.is_typed())
print("molecules :", set(site.molecule.name for site in topology.sites))

Angles and dihedrals are zero because we have not asked GMSO to find and populate them yet.
That happens during `apply` when you pass `identify_connections=True`.

Look at a site before typing. It has an element and a position (as defined from the mBuild `Compound`), and
`site.atom_type` is still `None`.

In [ ]:
site = topology.sites[0]
print(site.name, site.element.symbol, site.position)
print("atom_type:", site.atom_type)

## Loading force fields

A `gmso.ForceField` comes from an XML file. Passing a bare name looks the
force field up from installed plugins, which is how `"oplsaa"` resolves. Passing
a path loads that file.

Here we use OPLS-AA for hexane and a flexible SPC water model from a local
XML.

In [ ]:
oplsaa = ForceField("oplsaa")
spcfw = ForceField("files/spcfw.xml")

print(oplsaa.name, "has", len(oplsaa.atom_types), "atom types")
print(spcfw.name, "has", len(spcfw.atom_types), "atom types")

for name, atom_type in spcfw.atom_types.items():
    print(name, atom_type.definition, atom_type.charge, atom_type.parameters)

## Applying a force field

`apply` atom types every site and attaches bond, angle, and dihedral types.
Pass a dict keyed by molecule name to mix force fields in one system.

- `identify_connections=True` finds angles and dihedrals from the bond graph
- `speedup_by_moltag=True` types each unique molecule once and reuses the result

In [ ]:
apply(
    top=topology,
    forcefields={"water": spcfw, "hexane": oplsaa},
    identify_connections=True,
    speedup_by_moltag=True,
)

## Looking at the typed Topology

Same `gmso.Topology` object now much more information. Angles and dihedrals now exist, and every
site carries an `AtomType` with parameters, units, and a functional form. Additionally, the `Topology` also carries information about bond types, angle types, dihedral types and their parameters.

In [ ]:
print("sites     :", topology.n_sites)
print("bonds     :", topology.n_bonds)
print("angles    :", topology.n_angles)
print("dihedrals :", topology.n_dihedrals)
print("typed     :", topology.is_typed())

In [ ]:
print("=" * 12, "atom types", "=" * 12)
seen = set()
for site in topology.sites:
    if site.atom_type.name in seen:
        continue
    seen.add(site.atom_type.name)
    print(site.atom_type.name, site.atom_type.charge, site.atom_type.parameters)

print()
print("=" * 12, "expressions", "=" * 12)
print("nonbonded", topology.sites[0].atom_type.expression)
print("bond     ", topology.bonds[0].bond_type.expression)
print("angle    ", topology.angles[0].angle_type.expression)
print("dihedral ", topology.dihedrals[0].dihedral_type.expression)

In [ ]:
bond = topology.bonds[0]
members = "-".join(m.atom_type.name for m in bond.connection_members)
print(members, bond.bond_type.parameters)

angle = topology.angles[0]
members = "-".join(m.atom_type.name for m in angle.connection_members)
print(members, angle.angle_type.parameters)

GMSO also gives you a dataframe view, which is handy for a quick scan.

In [ ]:
topology.to_dataframe(parameter="sites").head(5)

In [ ]:
topology.to_dataframe(parameter="bonds").head(5)

In [ ]:
topology.to_dataframe(parameter="angles").head(5)

In [ ]:
topology.to_dataframe(parameter="dihedrals").head(5)

## Writing simulation inputs
GMSO is designed to interface with multiple simulation engines. This primarly occurs through engine-specific wrtiers, that all operate on the same `gmso.Topology` object.

The GROMCS and LAMMPS writers produce their needed input files, and the HOOMD-Blue writer produces the Python objects (initial state, list of forces) needed to start an HOOMD simulation.

One thing to watch. The LAMMPS writer converts potentials in place to the forms
LAMMPS expects, so run it last if you want inputs for several engines from a
single topology. Otherwise rebuild the topology between writers.

In [ ]:
from gmso.formats import write_gro, write_top, write_lammpsdata
from gmso.external.convert_hoomd import to_gsd_snapshot, to_hoomd_forcefield

### GROMACS

Below, you will call the `write_gro` and `write_top` methods, each being given the same `gmso.Topology` object created above. These will write a `.gro` and `.top` file to disk. You can open these to view the entire file, but in the cells below you see the initial (head) portions of each.

In [ ]:
write_gro(topology, "water_hexane.gro")
write_top(topology, "water_hexane.top")

In [ ]:
!head -n 8 water_hexane.gro

In [ ]:
!head -n 30 water_hexane.top

### HOOMD-Blue

No input files here. You get a GSD snapshot and a dict of HOOMD force objects
that you can hand straight to a `hoomd.md.Integrator`.

In [ ]:
snapshot, snapshot_refs = to_gsd_snapshot(topology)
forces, force_refs = to_hoomd_forcefield(topology, r_cut=1.2)

print(snapshot.particles.N, "particles")
print(snapshot.particles.types)
for kind, force_list in forces.items():
    print(kind, [type(f).__name__ for f in force_list])

### LAMMPS

Here, you'll use the `write_lammpsdata` function to create a `.data` file. This writer takes in LAMMPS-specific options as parameters, such as `atom_style` and `unit_style`.

In [ ]:
write_lammpsdata(topology, "water_hexane.data", atom_style="full", unit_style="real")

In [ ]:
!head -n 30 water_hexane.data

<h1 style="color: green;">Exercise</h1>

Build a benzene and ethanol mixture and take it all the way to input files.
Unlike the system above these two actually mix, so pack them uniformly.

1. Load benzene from `files/benzene.mol2` and ethanol from a SMILES string
2. Give each a name, then pack them into a box with `mb.fill_box`
3. Convert to a Topology and apply OPLS-AA to both
4. Write a GROMACS `.top` and `.gro`

Tip: `mb.fill_box` takes a list of Compounds and a matching list of counts.

Tip: If you want to know more about how a function works, and what parameters it takes use Python's built in `help()` method. Example: Try running `help(mb.fill_box)` or `mb.fill_box?` in a cell below.

In [ ]:
# Your code here

<h2 style="color: blue;">Answer</h2>

Run the cell below to see one solution.

In [ ]:
benzene = mb.load("files/benzene.mol2")
benzene.name = "benzene"
ethanol = mb.load("CCO", smiles=True)
ethanol.name = "ethanol"

mixture = mb.fill_box(
    compound=[benzene, ethanol],
    n_compounds=[100, 100],
    box=[4.0, 4.0, 4.0],
)
mixture.visualize()

In [ ]:
mixture_top = mixture.to_gmso()
apply(
    top=mixture_top,
    forcefields=oplsaa,
    identify_connections=True,
    speedup_by_moltag=True,
)

write_gro(mixture_top, "mixture.gro")
write_top(mixture_top, "mixture.top")
print("typed:", mixture_top.is_typed(), "|", mixture_top.n_sites, "sites")

---

### Recap

- `mb.load` builds Compounds from SMILES strings or from files
- `mb.fill_box` packs uniformly, `mb.fill_region` gives each species its own
  sub-volume, which is how you get a layered system
- `Compound.to_gmso()` produces an untyped Topology
- `apply` atom types it, and accepts a dict so different species get different force fields
- One typed Topology writes inputs for LAMMPS, GROMACS, and HOOMD-Blue

Next up is the advanced notebook, where mBuild is the focus and you write your
own Compound classes and your own force field XML.